# LeetCode Problem Learnings Cheat Sheet

**Purpose**: Quick reference guide for key concepts, patterns, and solutions from LeetCode problems

**Problems Covered**:
1. p_28: Find the Index of the First Occurrence in a String (strStr)
2. Merge 2 Sorted Linked Lists
3. Remove Duplicates from Sorted Array
4. Remove Element
5. Valid Parentheses

---

## 1️⃣ String Matching: Find First Occurrence in String (p_28)

### Problem
Given two strings `needle` and `haystack`, return the index of the first occurrence of needle in haystack, or -1 if not found.

### Key Learnings

#### String Slicing for Comparison
```python
# Extract substring of exact length and compare
haystack[i:i+len(needle)] == needle
```

#### Critical Concept: The `+ 1` in Range
```python
# Without +1: range(len(haystack) - len(needle))
# Last possible match position: len(haystack) - len(needle)
# Since range() is exclusive at end, we MUST add +1 to include it

# Example: haystack = "abc" (length 3), needle = "c" (length 1)
# range(3 - 1 + 1) = range(3) = [0, 1, 2]  ✓ Includes position 2
# range(3 - 1)     = range(2) = [0, 1]     ✗ Misses position 2
```

#### Edge Cases
- **Empty needle**: Return 0 (by LeetCode convention)
- **Single character**: Works naturally with slicing
- **Needle longer than haystack**: Range becomes empty, returns -1
- **Multiple occurrences**: Returns first match, loop exits immediately

### Solution
```python
class Solution:
    def strStr(self, haystack: str, needle: str) -> int:
        if not needle:
            return 0
        
        # Iterate through all valid starting positions
        for i in range(len(haystack) - len(needle) + 1):
            if haystack[i:i+len(needle)] == needle:
                return i
        return -1
```

### Complexity
- **Time**: $O(n \cdot m)$ where $n$ = len(haystack), $m$ = len(needle)
- **Space**: $O(1)$ (just storing indices)

---

## 2️⃣ Linked List Manipulation: Merge Two Sorted Lists

### Problem
Merge two sorted linked lists into one sorted list.

### Key Learnings

#### Two Pointer Technique on Linked Lists
Instead of using indices like arrays, we use actual **node references**:
```python
pointer = merged        # Current position in new list
list1 = list1.next      # Move to next node in list1
list2 = list2.next      # Move to next node in list2
pointer = pointer.next  # Move position forward in merged list
```

#### Building a New Linked List
```python
# Create dummy node as starting point
merged = ListNode()
pointer = merged

# Attach nodes at pointer.next (don't overwrite pointer itself)
pointer.next = list1  # Attach comparison winner
pointer = pointer.next  # Move pointer forward

# Return merged.next (skip the dummy node)
return merged.next
```

#### Handle Remaining Elements
After one list exhausts, attach all remaining from the other:
```python
if list1:
    pointer.next = list1
elif list2:
    pointer.next = list2
```

### Solution
```python
class Solution:
    def mergeTwoLists(self, list1, list2):
        merged = ListNode()
        pointer = merged
        
        # Compare and attach smaller nodes
        while list1 and list2:
            if list1.val < list2.val:
                pointer.next = list1
                list1 = list1.next
            else:
                pointer.next = list2
                list2 = list2.next
            pointer = pointer.next
        
        # Attach remaining
        if list1:
            pointer.next = list1
        elif list2:
            pointer.next = list2
        
        return merged.next
```

### Complexity
- **Time**: $O(n + m)$ where $n$, $m$ are list lengths
- **Space**: $O(1)$ (not counting output list)

---

## 3️⃣ Two Pointer Pattern: Remove Duplicates from Sorted Array

### Problem
Remove duplicates from a sorted array **in-place** using $O(1)$ extra space. Modify the array so each unique element appears once.

### Key Learnings

#### Two Pointer Technique (Read/Write)
```
CONCEPT: One pointer reads, one pointer writes
- read:  Scans through the array looking for new values
- write: Marks where the next unique value should go

Initial state:
  write = 1  (first unique value goes at index 1)
  read  = 1  (start comparing from index 1)
```

#### Detection Logic
```python
if nums[read] != nums[read - 1]:  # New unique value found!
    nums[write] = nums[read]       # Write it to the front
    write += 1                     # Move write pointer forward
```

Why `read - 1`? Because array is sorted, a change means new unique value.

#### Return Value
Return `write` - the count of unique elements. The first `write` elements contain all unique values.

### Solution
```python
class Solution:
    def removeDuplicates(self, nums: List[int]) -> int:
        if not nums:
            return 0
        
        write = 1  # Position to write next unique value
        
        for read in range(1, len(nums)):
            if nums[read] != nums[read - 1]:  # New value found
                nums[write] = nums[read]       # Write to front
                write += 1
        
        return write
```

### Complexity
- **Time**: $O(n)$ single pass
- **Space**: $O(1)$ modify in-place

### Example Trace
```
Input: [1, 1, 2]
write=1, read=1: nums[1]=1 == nums[0]=1? Yes → skip
write=1, read=2: nums[2]=2 != nums[1]=1? Yes → nums[1]=2, write=2
Return: 2, Array becomes [1, 2, *]
```

---

## 4️⃣ Two Pointer Pattern: Remove Element

### Problem
Remove all occurrences of a value from an array **in-place** in $O(1)$ space. Return the count of remaining elements.

### Key Learnings

#### Similar Pattern, Different Logic
The **read/write pattern** remains the same as remove duplicates, but the condition changes:

```python
# Remove Duplicates logic:
if nums[read] != nums[read - 1]:  # Compare with previous

# Remove Element logic:
if nums[read] != val:              # Compare with target value
```

#### Why This Works
By only writing non-target values to the front of the array, all target values are effectively pushed to the back (outside our returned count).

### Solution
```python
class Solution:
    def removeElement(self, nums: List[int], val: int) -> int:
        if not nums:
            return 0
        
        write = 0  # Position to write non-val elements
        
        for read in range(len(nums)):
            if nums[read] != val:  # Found a value to keep
                nums[write] = nums[read]  # Write to front
                write += 1
        
        return write
```

### Complexity
- **Time**: $O(n)$ single pass
- **Space**: $O(1)$ modify in-place

### Example Trace
```
Input: [3, 2, 2, 3], val = 3
write=0: nums[0]=3==3? Skip
write=0: nums[1]=2≠3? Yes → nums[0]=2, write=1
write=1: nums[2]=2≠3? Yes → nums[1]=2, write=2
write=2: nums[3]=3==3? Skip
Return: 2, Array becomes [2, 2, *, *]
```

### Key Difference vs Remove Duplicates
| Aspect | Remove Duplicates | Remove Element |
|--------|------------------|-----------------|
| Condition | Compare with previous element | Compare with target value |
| Requires | Sorted array | Any array order |
| Write start | 1 | 0 |

---

## 5️⃣ Stack Pattern: Valid Parentheses

### Problem
Given a string containing parentheses, brackets, and braces, determine if they are valid (properly closed and nested).

### Key Learnings

#### Stack Data Structure for Matching
**Use case**: When you need to match pairs with nesting order
- **Push** opening brackets onto stack
- **Pop** and verify closing bracket matches when found

#### First Check: Length Must Be Even
```python
if len(s) % 2 != 0:
    return False  # Impossible to match odd number of characters
```

#### Algorithm Flow
```
1. For each character:
   - If opening bracket → push to stack
   - If closing bracket → pop from stack and check match
2. At end: stack must be empty (all matched)
```

#### Matching Logic
```python
opening = ['(', '{', '[']
closing = [')', '}', ']']

# When closing bracket found:
if not stack or opening.index(stack.pop()) != closing.index(p):
    return False  # Either stack empty or brackets don't match
```

### Solution
```python
class Solution:
    def isValid(self, s: str) -> bool:
        if len(s) % 2 != 0:
            return False
        
        stack = []
        opening = ['(', '{', '[']
        closing = [')', '}', ']']
        
        for char in s:
            if char in opening:
                stack.append(char)
            elif char in closing:
                if not stack or opening.index(stack.pop()) != closing.index(char):
                    return False
        
        return not stack  # Empty stack = all matched
```

### Complexity
- **Time**: $O(n)$ single pass, each char processed once
- **Space**: $O(n)$ worst case (all opening brackets)

### Example Trace
```
Input: "([])"
char='(': in opening → stack=['(']
char='[': in opening → stack=['(', '[']
char=']': closing_idx=2, pop '['  opening_idx=2 ✓ → stack=['(']
char=')': closing_idx=0, pop '('  opening_idx=0 ✓ → stack=[]
End: stack empty → return True ✓
```

### Common Mistake
```python
# ❌ WRONG: Checking only first character
if i == needle[0]:  # This compares index to character!

# ✓ RIGHT: Check the character at that index
if haystack[i] == needle[0]  # Compare characters
```

---

## 🎯 Pattern Recognition Reference

### Recurring Patterns Across Problems

| Pattern | Use When | Example |
|---------|----------|---------|
| **Iteration + Slicing** | Need to check all substrings or windows | strStr (substring matching) |
| **Two Pointers (Read/Write)** | Modifying array in-place with $O(1)$ space | Remove duplicates, remove element |
| **Linked List Traversal** | Combining/merging linked lists | Merge sorted lists |
| **Stack (LIFO)** | Matching pairs with nesting order | Valid parentheses |
| **Early Returns** | Found target or impossible condition | Return immediately, don't continue |
| **Edge Case Checks** | Before main logic to prevent errors | Empty strings, odd lengths, boundary conditions |

### When to Use Each Data Structure

```python
# STRINGS: Use when you need substring comparisons
haystack[i:i+len(needle)]  # Extract substring

# LINKED LISTS: Keep references to nodes, not indices
pointer = pointer.next

# ARRAYS: Good for two-pointer techniques IN-PLACE
read, write = operations on indices

# STACKS: Need LIFO behavior or matching pairs
stack.append() / stack.pop()
```

---

## 📊 Time & Space Complexity Cheat Sheet

### Quick Reference

| Problem | Time | Space | Notes |
|---------|------|-------|-------|
| **strStr** | $O(n \cdot m)$ | $O(1)$ | $n$ = haystack, $m$ = needle. Brute force comparison. |
| **Merge Sorted Lists** | $O(n + m)$ | $O(1)$ | $n$, $m$ = list lengths. Each node visited once. |
| **Remove Duplicates** | $O(n)$ | $O(1)$ | Single pass, in-place modification. |
| **Remove Element** | $O(n)$ | $O(1)$ | Single pass, in-place modification. |
| **Valid Parentheses** | $O(n)$ | $O(n)$ | Stack stores opening brackets in worst case. |

### Complexity Analysis Tips

```
TIME COMPLEXITY:
- Single pass through array/string → O(n)
- Nested loops (i through length, j through substring) → O(n*m)
- Stack operations (push/pop) → O(1) each, O(n) total if done n times

SPACE COMPLEXITY:
- In-place modifications → O(1) extra space
- Using extra stack/list → O(n) proportional to input
- Not counting output in space calculation
```

### Why $O(n \cdot m)$ for strStr?
```python
for i in range(len(haystack) - len(needle) + 1):        # Up to n iterations
    if haystack[i:i+len(needle)] == needle:              # Slice & compare: O(m)
        return i
# Total: O(n) * O(m) = O(n*m)
```

---

## ⚠️ Common Pitfalls & Solutions

### 1. Index vs Character Comparison
```python
# ❌ WRONG
if i == needle[0]:  # Comparing index (int) with character (str)

# ✓ CORRECT
if haystack[i] == needle[0]  # Compare characters
```

### 2. Off-by-One in Range
```python
# ❌ WRONG: Misses last valid starting position
for i in range(len(haystack) - len(needle)):

# ✓ CORRECT: Includes last valid position
for i in range(len(haystack) - len(needle) + 1):
```

### 3. Two-Pointer Write Position
```python
# ❌ WRONG: Starting write = 0 for remove duplicates
remove_dupes: write = 0  # First element always unique!

# ✓ CORRECT: Start at 1 if checking duplicates
remove_dupes: write = 1  # Compare with previous
remove_element: write = 0  # No previous comparison needed
```

### 4. Forgetting Stack State Check
```python
# ❌ WRONG: Only pops, doesn't check if empty
if opening.index(stack.pop()) != closing.index(p):

# ✓ CORRECT: Check empty AND match
if not stack or opening.index(stack.pop()) != closing.index(p):
```

### 5. Linked List Dummy Node
```python
# ❌ WRONG: Losing the start of new list
merged = ListNode()
return merged  # This is the dummy, not the real list!

# ✓ CORRECT: Skip dummy when returning
merged = ListNode()
pointer = merged
# ...build list...
return merged.next  # Skip dummy node
```

### 6. Edge Cases to Always Check
- Empty inputs
- Single elements
- Inputs longer than expected (needle > haystack)
- All elements matching (all zeros, all duplicates)
- No matches at all

---

## 📝 Quick Study Summary

### Key Takeaways
1. **String operations** use iteration + slicing for substring matching
2. **Two-pointer technique** is powerful for in-place array modifications
3. **Linked lists** require tracking node references, not indices
4. **Stacks** solve matching/pairing problems elegantly
5. **Edge cases** are where most bugs hide—check them first

### Before Coding
- [ ] Identify the problem pattern (string? array? linked list? pairs?)
- [ ] Plan edge cases
- [ ] Write accurate complexity estimate
- [ ] Start with pseudocode/comments

### While Coding
- [ ] Use meaningful variable names (`write`, `read`, `pointer`, `stack`)
- [ ] Add comments explaining non-obvious logic
- [ ] Test edge cases mentally before running

### After Coding
- [ ] Trace through 1-2 examples by hand
- [ ] Verify complexity matches your plan
- [ ] Check boundary conditions

---

**Happy coding! 🚀 Reference this guide when stuck on similar problems.**